# GFED-based GDSTEM grid-cell selection

This reference notebook reorganizes `03_select_gridcells.py` into a sequential workflow, following the structure of the local `fluxnet-gdstem-site-selection.ipynb`: English explanations, numbered steps, code cells, and table previews.

It selects up to three observation-based calibration cells per enabled TEM community:

1. validate completed Stage 1 and Stage 2 inputs;
2. retain cells with dominant, stable community fractions;
3. combine the mapped GFED classes and count distinct fire years;
4. screen for complete GFED coverage and sufficient fire activity;
5. rank candidates for low, medium, and high fire;
6. apply optional manual overrides and report every community from 1 to 23.

The original source file is unchanged. The notebook has not been run against the server data. Review the paths below, then run all cells from top to bottom in a fresh kernel. If you change configuration, restart the kernel and run all cells again.

The TEM/GFED mappings are inherited from the source script. They are proposed compatibilities or proxies, rather than newly validated scientific equivalences.


## Configuration and input paths

Only edit this section for a normal run. The default paths refer to the original server. `RESULT_DIR` is a separate notebook output directory, so running this reference version does not overwrite the original Stage 3 results. Reusing the notebook result directory replaces its result tables; its manual override file is preserved.


In [ ]:
import json
import sqlite3
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# Configuration: edit paths and settings in this section.

# Output directory used by Stages 1 and 2. Update this path for your server.
OUTPUT_DIR = Path('/quobyte/emoniergrp/luyy123/calibration_gridcell_selection_2002_2022_v2')

# Use the same inclusive year range as Stages 1 and 2: 2002-2022 = 21 years.
START_YEAR = 2002
END_YEAR = 2022
N_YEARS = END_YEAR - START_YEAR + 1

# Read the full_1_23 Stage 1 files, rather than the earlier communities 1-14 files.
FRAC_FILE = OUTPUT_DIR / f'01_community_fraction_{START_YEAR}_{END_YEAR}_full_1_23.csv'
STAGE1_RUN_SUMMARY = OUTPUT_DIR / f'01_run_summary_{START_YEAR}_{END_YEAR}_full_1_23.json'

# Stage 2 outputs must cover every grid cell required by the current Stage 1 data.
GFED_MEAN = OUTPUT_DIR / f'02_gfed_by_class_summary_{START_YEAR}_{END_YEAR}.csv'
GFED_ANNUAL = OUTPUT_DIR / f'02_gfed_annual_nonzero_{START_YEAR}_{END_YEAR}.csv'

# Write Stage 3 outputs to a separate subdirectory.
# Change RESULT_DIR before each run if you want to keep multiple trials.
# Reusing RESULT_DIR replaces result tables and JSON but preserves manual overrides.
RESULT_DIR = OUTPUT_DIR / f'stage3_full_1_23_{START_YEAR}_{END_YEAR}_notebook'

# Location for a temporary SQLite database; use scratch storage if needed.
# Each run creates and removes its own temporary directory.
WORK_DIR = OUTPUT_DIR

# Number of CSV rows read at a time; this does not change the calculations.
# Eligible candidates and matched GFED summary records are still retained in memory.
CSV_CHUNKSIZE = 150_000

# Original community dominance and stability thresholds.
MIN_MEAN_COMMUNITY_FRACTION = 0.60
MIN_MIN_COMMUNITY_FRACTION = 0.40
MIN_FIRE_YEARS = 3

# Select low, medium, then high fire, without reusing a cell within a community.
# Targets are the 30th/50th/70th percentiles; bin boundaries remain 40% and 60%.
TARGET_QUANTILES = {'low_fire': 0.30, 'medium_fire': 0.50, 'high_fire': 0.70}
SHORTLIST_K = 20

# By default, require a complete Stage 1 run with no unclassified area.
# Set True only if Stage 1 allows unclassified area in the total-area denominator.
ALLOW_STAGE1_UNCLASSIFIED_AREA = False

# Flag annual burned area greater than the whole grid-cell area for review.
# False: flag only; do not remove candidates, cap values, or change selection.
# True: exclude flagged candidates before recalculating percentiles and selections.
# A flag alone does not establish a product error; repeated burning needs review.
EXCLUDE_QC_FLAGGED_CANDIDATES = False
ANNUAL_AREA_REVIEW_RATIO = 1.0


## Classification mappings

All 23 communities remain registered. Empty GFED tuples disable selection for communities 1, 15, and 23. The supplied grassland and cropland proxies are retained, together with their limitations. An approximation flag of `False` does not prove exact equivalence.

**Correction to the source:** `mapping_info()` previously returned one string, while `crosswalk_table()` expected `status, note`. It now returns both fields. The community IDs and GFED class assignments are unchanged.


In [ ]:
# Preserve the community IDs supplied in the original script.
COMMUNITY_NAMES = {
    1: 'Ice', 2: 'Alpine Tundra and Polar Desert', 3: 'Moist and Wet Tundra',
    4: 'Boreal Forests', 5: 'Temperate Coniferous Forests',
    6: 'Temperate Deciduous Forests', 7: 'Temperate Broadleaved Evergreen Forest',
    8: 'Short Grassland', 9: 'Tall Grassland', 10: 'Mediterranean Shrubland',
    11: 'Xeric Shrubland', 12: 'Xeric Woodland',
    13: 'Tropical Evergreen Forests', 14: 'Tropical Deciduous Forests',
    15: 'Impervious Surfaces', 16: 'Turf Lawn', 17: 'Soybean', 18: 'Maize',
    19: 'Fruit tree', 20: 'Switchgrass', 21: 'Vegetable Farm',
    22: 'Pasture', 23: 'Rangeland',
}

GFED_NAMES = {
    1: 'Tundra', 2: 'Sparse boreal forest', 3: 'Boreal forest',
    4: 'Temperate grassland', 5: 'Temperate shrubland', 6: 'Temperate mosaic',
    7: 'Temperate forest', 8: 'Tropical grassland', 9: 'Tropical shrubland',
    10: 'Open savanna', 11: 'Woody savanna', 12: 'Tropical forest',
    13: 'Other', 14: 'Cropland', 15: 'Peat', 16: 'Deforestation',
}

# Register all 23 communities; an empty tuple disables GFED-based selection.
# Communities 2-14 retain the proposed compatibility mapping, not an official crosswalk.
# Communities 16 and 22 retain the supplied GFED 4+8 grassland proxy.
# Communities 17-21 use generic Cropland; crop-specific fire cannot be separated.
# Communities 19/20 assume managed orchards/energy crops; review if definitions change.
# Communities 1 and 15 have no direct match; do not assign them to Other.
# Community 23 remains deferred, as configured in the original script.
COMMUNITY_TO_GFED = {
    1: (),
    2: (1,), 3: (1,), 4: (2, 3), 5: (7,), 6: (7,), 7: (7,),
    8: (4, 8), 9: (4, 8), 10: (5,), 11: (5, 9),
    12: (6, 10, 11), 13: (12,), 14: (12,),
    15: (),
    16: (4, 8),
    17: (14,), 18: (14,), 19: (14,), 20: (14,), 21: (14,),
    22: (4, 8),
    23: (),
}

# Preserve the original approximation flags and flag every added proxy.
# False does not imply exact equivalence; communities 2-14 remain proposed matches.
APPROXIMATE_CROSSWALK = {4, 8, 9, 11, 12, 14, 16, 17, 18, 19, 20, 21, 22}

# Shared keys and required Stage 1 columns.

KEY = ['lon', 'lat', 'community_type']
FRACTION_COLUMNS = [
    'lon', 'lat', 'community_type', 'community_name',
    'mean_community_fraction', 'min_community_fraction', 'max_community_fraction',
    'n_years_grid',
]




def mapping_info(comm):
    """Return a mapping status and explanatory note for the output crosswalk."""
    if comm in (1, 15):
        return 'no_direct_match', 'No direct GFED class is assigned; selection is disabled.'
    if comm == 23:
        return 'deferred_by_user', 'Selection remains deferred as specified in the source configuration.'
    if comm in (16, 22):
        return ('user_confirmed_grassland_proxy',
                'GFED classes 4 and 8 are proxies; turf or pasture fire cannot be isolated.')
    if 17 <= comm <= 21:
        return ('proposed_generic_cropland_proxy',
                'GFED Cropland is a generic proxy and cannot isolate individual crops.')
    return ('original_proposed_compatibility',
            'Proposed compatibility retained from the source; not an official one-to-one crosswalk.')


def crosswalk_table():
    """Build one crosswalk row for each of the 23 communities."""
    rows = []
    for comm in range(1, 24):
        ids = COMMUNITY_TO_GFED[comm]
        status, note = mapping_info(comm)
        approximate = None
        if ids:
            approximate = comm in APPROXIMATE_CROSSWALK
        rows.append({
            'community_type': comm, 'community_name': COMMUNITY_NAMES[comm],
            'selection_enabled': bool(ids),
            'gfed_class_ids': ','.join([str(gfed_id) for gfed_id in ids]),
            'gfed_class_names': '; '.join(GFED_NAMES[g] for g in ids),
            'mapping_approximate': approximate,
            'mapping_status': status, 'mapping_note': note,
        })
    return pd.DataFrame(rows)

crosswalk = crosswalk_table()
display(crosswalk)


## 1. Validate upstream outputs and initialize this run

Stage 1 must have the configured year range, point to the selected fraction CSV, and have an accepted completion status. Stage 2 rows are checked when read in the following steps.

The run summary is saved after every processing step and records ordinary Python exceptions as failures. If you stop between cells, it remains `running`; kernel termination cannot be recorded as a completed or failed run.


In [ ]:
def output_paths():
    """Return the paths for all Stage 3 outputs."""
    return {
        'crosswalk': RESULT_DIR / '03_community_gfed_crosswalk.csv',
        'coverage': RESULT_DIR / '03_missing_gfed_summary_rows.csv',
        'qc': RESULT_DIR / '03_gfed_area_review.csv',
        'candidates': RESULT_DIR / '03_all_eligible_candidates.csv',
        'shortlist': RESULT_DIR / '03_ranked_shortlists.csv',
        'selected': RESULT_DIR / '03_selected_observation_only.csv',
        'summary': RESULT_DIR / '03_community_screening_summary.csv',
        'overrides': RESULT_DIR / '03_manual_overrides.csv',
        'manifest': RESULT_DIR / '03_run_summary.json',
    }


def check_inputs():
    """Validate settings and require completed upstream files before reading candidates."""
    if START_YEAR > END_YEAR or CSV_CHUNKSIZE <= 0 or SHORTLIST_K < 1:
        raise ValueError('Invalid year range, CSV chunk size, or shortlist length.')
    if not (1 <= MIN_FIRE_YEARS <= N_YEARS):
        raise ValueError('MIN_FIRE_YEARS must be between 1 and N_YEARS.')
    if set(COMMUNITY_NAMES) != set(range(1, 24)) or set(COMMUNITY_TO_GFED) != set(range(1, 24)):
        raise ValueError('Both community tables must contain every ID from 1 to 23.')
    for ids in COMMUNITY_TO_GFED.values():
        if len(ids) != len(set(ids)) or not set(ids).issubset(GFED_NAMES):
            raise ValueError('The GFED mapping contains duplicate or unknown class IDs.')
    for path in (FRAC_FILE, STAGE1_RUN_SUMMARY, GFED_MEAN, GFED_ANNUAL):
        if not path.is_file():
            raise FileNotFoundError(f'Required upstream output is missing: {path}')
    stage1 = json.loads(STAGE1_RUN_SUMMARY.read_text(encoding='utf-8'))
    allowed = {'complete'}
    if ALLOW_STAGE1_UNCLASSIFIED_AREA:
        allowed.add('complete_with_unclassified_area')
    if stage1.get('status') not in allowed:
        raise ValueError(f"Stage 1 status is {stage1.get('status')}; finish Stage 1 or review unclassified area.")
    source = stage1.get('source', {})
    if (source.get('start_year'), source.get('end_year')) != (START_YEAR, END_YEAR):
        raise ValueError('The Stage 1 year range does not match the Stage 3 settings.')
    if Path(stage1.get('paths', {}).get('fractions', '')).resolve() != FRAC_FILE.resolve():
        raise ValueError('The Stage 1 completion record points to a different fraction file.')
    return stage1


def save_run_summary(manifest, paths):
    """Save the current run status and output locations."""
    text = json.dumps(manifest, ensure_ascii=False, indent=2)
    paths['manifest'].write_text(text, encoding='utf-8')


stage1 = check_inputs()
RESULT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
paths = output_paths()
manifest = {
    'status': 'running', 'start_year': START_YEAR, 'end_year': END_YEAR,
    'stage1_status': stage1['status'],
    'inputs': {'fractions': str(FRAC_FILE), 'gfed_mean': str(GFED_MEAN),
               'gfed_annual': str(GFED_ANNUAL)},
    'outputs': {k: str(v) for k, v in paths.items()},
    'exclude_qc_flagged_candidates': EXCLUDE_QC_FLAGGED_CANDIDATES,
    'annual_area_review_ratio': ANNUAL_AREA_REVIEW_RATIO,
    'thresholds': {'mean_fraction': MIN_MEAN_COMMUNITY_FRACTION,
                   'min_fraction': MIN_MIN_COMMUNITY_FRACTION,
                   'min_fire_years': MIN_FIRE_YEARS,
                   'target_quantiles': TARGET_QUANTILES, 'shortlist_k': SHORTLIST_K},
    'crosswalk': crosswalk_table().to_dict('records'),
}

save_run_summary(manifest, paths)
print('Validated upstream files. Output directory:', RESULT_DIR)


## 2. Screen community dominance and stability

Read the Stage 1 fraction table in chunks. A candidate must have all configured years, a mean community fraction of at least 0.60, and a minimum annual community fraction of at least 0.40. Only communities with an assigned GFED mapping enter the candidate table.

Chunked reading limits input-buffer size. Candidate records are still stored in memory. `total` and `passed` retain counts for all communities, including those without a GFED mapping.


In [ ]:
def read_eligible_community():
    """Read fraction records in chunks, count each community, and retain eligible candidates."""
    total = {c: 0 for c in COMMUNITY_NAMES}
    passed = {c: 0 for c in COMMUNITY_NAMES}
    parts = []
    enabled = {c for c, ids in COMMUNITY_TO_GFED.items() if ids}
    for chunk in pd.read_csv(FRAC_FILE, chunksize=CSV_CHUNKSIZE):
        missing = set(FRACTION_COLUMNS) - set(chunk.columns)
        if missing:
            raise ValueError(f'Stage 1 is missing columns: {sorted(missing)}')
        if not chunk.community_type.isin(COMMUNITY_NAMES).all():
            raise ValueError('Stage 1 contains missing community IDs or IDs outside 1-23.')
        chunk['lon'] = chunk.lon.round(6)
        chunk['lat'] = chunk.lat.round(6)
        # Use the supplied names; Stage 1 has already classified currentveg/agstate.
        chunk['community_name'] = chunk.community_type.map(COMMUNITY_NAMES)
        ok = ((chunk.n_years_grid == N_YEARS) &
              (chunk.mean_community_fraction >= MIN_MEAN_COMMUNITY_FRACTION) &
              (chunk.min_community_fraction >= MIN_MIN_COMMUNITY_FRACTION))
        for c, n in chunk.community_type.value_counts().items():
            total[int(c)] += int(n)
        for c, n in chunk.loc[ok, 'community_type'].value_counts().items():
            passed[int(c)] += int(n)
        use = ok & chunk.community_type.isin(enabled)
        if use.any():
            parts.append(chunk.loc[use].copy())
    eligible = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=FRACTION_COLUMNS)
    if eligible.duplicated(KEY).any():
        raise ValueError('Stage 1 contains duplicate grid-cell/community rows; check the input.')
    print(f'LULCC records: {sum(total.values()):,}; mapped candidates passing the fraction screen: {len(eligible):,}', flush=True)
    return eligible, total, passed

try:
    crosswalk = crosswalk_table()
    crosswalk.to_csv(paths['crosswalk'], index=False)
    # Clear the coverage report even when no candidates pass the community screen.
    pd.DataFrame(columns=KEY + ['gfed_id']).to_csv(paths['coverage'], index=False)
    eligible, total, passed = read_eligible_community()
except Exception as exc:
    manifest.update(status='failed', error=str(exc))
    save_run_summary(manifest, paths)
    raise
else:
    save_run_summary(manifest, paths)

display(eligible.head())


## 3. Combine GFED classes and count distinct fire years

For each candidate, sum burned area across its mapped GFED classes. The fire-activity metric is:

`gfed_mean_grid_fraction = sum of mapped burned areas / N_YEARS / whole grid-cell area`

The denominator is the whole grid-cell area. A year with fire in two mapped classes counts as **one** fire year. Missing GFED summary rows stop selection and are listed in the coverage report; they are not treated as zeros.

The temporary SQLite database is retained to aggregate annual records on disk across CSV chunks. Its unique key is `(lon, lat, community_type, year)`. The `ON CONFLICT` statement adds area to an existing key. Replacing this with one large in-memory table could substantially increase memory use.

For complete records, the summed annual areas must agree with the multi-year summary. Annual area above the configured grid-area ratio receives a review flag. By default, this flag does not exclude the candidate.


In [ ]:
def build_links(eligible):
    """Create one link for each required grid-cell/community/GFED-class combination."""
    parts = []
    for comm, ids in COMMUNITY_TO_GFED.items():
        src = eligible.loc[eligible.community_type == comm, KEY]
        for gfed in ids:
            x = src.copy()
            x['gfed_id'] = gfed
            parts.append(x)
    return pd.concat(parts, ignore_index=True)


def joined_gfed_summary(links, paths):
    """Sum mapped GFED classes and divide by the year count and whole grid-cell area."""
    parts = []
    need = ['lon', 'lat', 'gfed_id', 'gfed_valid_years', 'gfed_ba_sum_km2', 'grid_area_km2']
    for chunk in pd.read_csv(GFED_MEAN, usecols=need, chunksize=CSV_CHUNKSIZE):
        chunk['lon'] = chunk.lon.round(6)
        chunk['lat'] = chunk.lat.round(6)
        m = chunk.merge(links, on=['lon', 'lat', 'gfed_id'], how='inner')
        if not m.empty:
            parts.append(m)
    matched = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=need + ['community_type'])
    if matched.duplicated(KEY + ['gfed_id']).any():
        raise ValueError('Stage 2 summary contains duplicate grid-cell/GFED-class records.')
    # A missing Stage 2 row differs from an existing row with zero burned area.
    coverage = links.merge(matched[KEY + ['gfed_id']], on=KEY + ['gfed_id'],
                           how='left', indicator=True, validate='one_to_one')
    absent = coverage.loc[coverage['_merge'] == 'left_only', KEY + ['gfed_id']]
    absent.to_csv(paths['coverage'], index=False)
    if not absent.empty:
        raise ValueError(f"Stage 2 summary is missing {len(absent)} required grid-cell/class rows; "
                         f"see {paths['coverage']} and complete or rerun Stage 2 before selection.")
    vals = matched[['gfed_ba_sum_km2', 'grid_area_km2', 'gfed_valid_years']].to_numpy(dtype=float)
    if (not np.isfinite(vals).all() or (matched.gfed_ba_sum_km2 < 0).any() or
            (matched.grid_area_km2 <= 0).any() or
            not matched.gfed_valid_years.between(0, N_YEARS).all() or
            (matched.gfed_valid_years % 1 != 0).any()):
        raise ValueError('Stage 2 summary contains invalid areas or valid-year counts.')
    if (matched.groupby(KEY).grid_area_km2.nunique() != 1).any():
        raise ValueError('Whole grid-cell areas differ between GFED classes for the same candidate.')
    obs = matched.groupby(KEY, as_index=False).agg(
        gfed_classes_found=('gfed_id', 'nunique'),
        gfed_min_valid_years=('gfed_valid_years', 'min'),
        gfed_max_valid_years=('gfed_valid_years', 'max'),
        gfed_ba_total_km2=('gfed_ba_sum_km2', 'sum'),
        grid_area_km2=('grid_area_km2', 'first'))
    # Retain the legacy 21y column name for compatibility; use the general field otherwise.
    obs['gfed_ba_total_21y_km2'] = obs.gfed_ba_total_km2
    obs['gfed_mean_ba_km2'] = obs.gfed_ba_total_km2 / N_YEARS
    obs['gfed_mean_grid_fraction'] = obs.gfed_mean_ba_km2 / obs.grid_area_km2
    return obs


def nonzero_years_for_communities(links):
    """Sum mapped classes within each year, then count distinct years with fire."""
    # One record per lon/lat/community/year: classes 4 and 8 burning in one year count once.
    # This temporary database is separate from the Stage 1 database and is removed afterward.
    with tempfile.TemporaryDirectory(prefix='stage3_full_1_23_', dir=WORK_DIR) as work:
        conn = sqlite3.connect(Path(work) / 'annual_fire.sqlite')
        try:
            # Limit the SQLite page cache to approximately 16 MB.
            conn.execute('PRAGMA cache_size=-16000')
            conn.execute('''CREATE TABLE fire_year (
                lon REAL,lat REAL,community_type INTEGER,year INTEGER,area REAL,
                PRIMARY KEY (lon,lat,community_type,year)) WITHOUT ROWID''')
            # Insert a new cell/community/year, or add area when that key already exists.
            # This also combines contributions spread across CSV chunks.
            query = '''INSERT INTO fire_year VALUES (?,?,?,?,?)
                ON CONFLICT(lon,lat,community_type,year)
                DO UPDATE SET area=area+excluded.area'''
            need = ['lon', 'lat', 'year', 'gfed_id', 'burned_area_km2']
            for i, chunk in enumerate(pd.read_csv(GFED_ANNUAL, usecols=need,
                                                 chunksize=CSV_CHUNKSIZE), 1):
                if (not chunk.year.between(START_YEAR, END_YEAR).all() or
                        (chunk.year % 1 != 0).any()):
                    raise ValueError('Stage 2 annual data contain invalid years or years outside the configured range.')
                areas = chunk.burned_area_km2.to_numpy(dtype=float)
                if not np.isfinite(areas).all() or (areas <= 0).any():
                    raise ValueError('Stage 2 annual_nonzero must contain only finite, positive burned areas.')
                chunk['lon'] = chunk.lon.round(6)
                chunk['lat'] = chunk.lat.round(6)
                merged = chunk.merge(links, on=['lon', 'lat', 'gfed_id'], how='inner')
                if not merged.empty:
                    group = merged.groupby(KEY + ['year'], as_index=False).burned_area_km2.sum()
                    rows = group[KEY + ['year', 'burned_area_km2']].itertuples(index=False, name=None)
                    with conn:
                        conn.executemany(query, rows)
                if i % 40 == 0:
                    print(f'  GFED annual chunks processed: {i}', flush=True)
            return pd.read_sql_query('''SELECT lon,lat,community_type,
                COUNT(*) AS gfed_nonzero_years,
                MAX(area) AS gfed_max_annual_ba_km2,
                SUM(area) AS gfed_annual_check_sum_km2
                FROM fire_year WHERE area>0 GROUP BY lon,lat,community_type''', conn)
        finally:
            conn.close()

try:
    if eligible.empty:
        merged = eligible.copy()
        full = pd.Series(False, index=merged.index, dtype=bool)
        qc_pass = pd.Series(True, index=merged.index, dtype=bool)
    else:
        links = build_links(eligible)
        obs = joined_gfed_summary(links, paths)
        freq = nonzero_years_for_communities(links)
        merged = eligible.merge(obs, on=KEY, validate='one_to_one').merge(
            freq, on=KEY, how='left', validate='one_to_one')
        merged['gfed_nonzero_years'] = pd.to_numeric(merged.gfed_nonzero_years).fillna(0).astype(int)
        for col in ('gfed_max_annual_ba_km2', 'gfed_annual_check_sum_km2'):
            merged[col] = pd.to_numeric(merged[col]).fillna(0)
        expected_classes = {}
        for community, classes in COMMUNITY_TO_GFED.items():
            expected_classes[community] = len(classes)
        merged['expected_gfed_classes'] = merged['community_type'].map(expected_classes)
        full = ((merged.gfed_classes_found == merged.expected_gfed_classes) &
                (merged.gfed_min_valid_years == N_YEARS) &
                (merged.gfed_max_valid_years == N_YEARS))
        # For complete records, annual positive areas must reproduce the multi-year total.
        consistent = np.isclose(merged.gfed_annual_check_sum_km2, merged.gfed_ba_total_km2,
                                rtol=1e-8, atol=1e-7)
        if (full & ~consistent).any():
            raise ValueError('GFED annual and summary areas disagree; check for mixed upstream runs.')
        if (merged.gfed_nonzero_years > N_YEARS).any():
            raise ValueError('The number of fire years exceeds the configured number of years.')
        merged['gfed_max_annual_grid_fraction'] = merged.gfed_max_annual_ba_km2 / merged.grid_area_km2
        merged['gfed_area_needs_review'] = (merged.gfed_max_annual_grid_fraction > ANNUAL_AREA_REVIEW_RATIO)
        merged['community_fraction_range'] = merged.max_community_fraction - merged.min_community_fraction
        merged = merged.merge(crosswalk.drop(columns=['community_name']),
                              on='community_type', how='left', validate='many_to_one')
        qc_pass = ~merged.gfed_area_needs_review
        # Flagged records remain eligible unless optional QC exclusion is enabled.
except Exception as exc:
    manifest.update(status='failed', error=str(exc))
    save_run_summary(manifest, paths)
    raise
else:
    save_run_summary(manifest, paths)

display(merged.head())


## 4. Screen fire activity and calculate community percentiles

Require complete GFED coverage, at least three distinct fire years, and a positive mean burned fraction. Exclude review-flagged candidates only when `EXCLUDE_QC_FLAGGED_CANDIDATES` is `True`.

Within each community, sort by mean burned fraction, longitude, and latitude. Assign empirical plotting positions `(i + 0.5) / n`, where `i` starts at zero. Tied burned fractions can therefore have different positions, matching the original method. These positions define the fire bins; pandas `quantile()` computes the target values separately.


In [ ]:
try:
    if eligible.empty:
        final = merged.copy()
    else:
        active = ((merged.gfed_nonzero_years >= MIN_FIRE_YEARS) &
                  (merged.gfed_mean_grid_fraction > 0))
        use = full & active
        if EXCLUDE_QC_FLAGGED_CANDIDATES:
            use &= qc_pass
        final = merged.loc[use].copy()
    # Calculate empirical percentiles separately within each community.
    if not final.empty:
        final = final.sort_values(['community_type', 'gfed_mean_grid_fraction', 'lon', 'lat']).copy()
        for comm, group in final.groupby('community_type', sort=True):
            final.loc[group.index, 'empirical_percentile'] = (np.arange(len(group)) + .5) / len(group)
    else:
        final['empirical_percentile'] = pd.Series(dtype=float)
    final.to_csv(paths['candidates'], index=False)
    qc_rows = merged.loc[~qc_pass].copy()
    qc_rows.to_csv(paths['qc'], index=False)
except Exception as exc:
    manifest.update(status='failed', error=str(exc))
    save_run_summary(manifest, paths)
    raise
else:
    save_run_summary(manifest, paths)

display(final.head())


## 5. Rank candidates and select low, medium, and high fire cells

Process the groups in this order:

| Group | Target quantile | Empirical-percentile bin |
| --- | --- | --- |
| Low fire | 0.30 | less than 0.40 |
| Medium fire | 0.50 | 0.40 to less than 0.60 |
| High fire | 0.70 | at least 0.60 |

Calculate each target from all eligible candidates for that community. Remove already selected coordinates, then use the requested bin. If the bin has no unused candidates, fall back to all unused candidates and record a flag.

Keep at most 20 candidates closest to the target fire fraction using `nsmallest()`, preserving the source's tie handling. Rank that shortlist by mean community fraction (descending), minimum fraction (descending), fraction range (ascending), distance to target (ascending), longitude, and latitude. These are sequential sorting criteria, not a weighted score.

The explicit coordinate loop replaces `apply(lambda ..., axis=1)`. A cell may be used by different communities, but cannot be reused within one community. A community with fewer than three available cells receives fewer selections.


In [ ]:
def select_group(candidates, community, label, q, selected_pairs):
    """Select one point and retain its ranked shortlist using the original rules.

    community is retained for compatibility with the original function signature.
    candidates must contain only this community's eligible records.
    """
    # Define the target from ALL eligible candidates, including previously selected cells.
    target = float(candidates['gfed_mean_grid_fraction'].quantile(q))

    # Explicit loops replace the row-wise apply(lambda ...) expression.
    available = []
    for lon, lat in zip(candidates['lon'], candidates['lat']):
        pair = (float(lon), float(lat))
        available.append(pair not in selected_pairs)
    subset = candidates.loc[available].copy()
    if subset.empty:
        return None, pd.DataFrame()

    percentile = subset['empirical_percentile']
    if label == 'low_fire':
        pool = subset.loc[percentile < 0.40].copy()
    elif label == 'medium_fire':
        pool = subset.loc[(percentile >= 0.40) & (percentile < 0.60)].copy()
    else:
        pool = subset.loc[percentile >= 0.60].copy()

    # If no unused candidate remains in this bin, use all unused candidates.
    fallback = False
    if pool.empty:
        pool = subset.copy()
        fallback = True

    pool['target_fire_fraction'] = target
    pool['distance_to_target'] = abs(pool['gfed_mean_grid_fraction'] - target)
    pool['distance_to_target_normalized'] = pool['distance_to_target'] / max(target, 1e-12)

    # Keep nsmallest to preserve its original handling of ties at the shortlist boundary.
    # Distance here means fire-fraction difference, not geographic distance.
    shortlist_size = min(SHORTLIST_K, len(pool))
    pool = pool.nsmallest(shortlist_size, 'distance_to_target').copy()
    pool['community_fraction_range'] = (
        pool['max_community_fraction'] - pool['min_community_fraction']
    )

    # Compare fields in order; these criteria are not combined into a weighted score.
    ranking_columns = [
        'mean_community_fraction', 'min_community_fraction',
        'community_fraction_range', 'distance_to_target', 'lon', 'lat',
    ]
    ranking_ascending = [False, False, True, True, True, True]
    pool = pool.sort_values(ranking_columns, ascending=ranking_ascending).copy()
    pool['fire_group'] = label
    pool['target_quantile'] = q
    pool['shortlist_rank'] = np.arange(1, len(pool) + 1)
    pool['fallback_no_quantile_bin'] = fallback
    best = pool.head(1).copy()
    return best, pool


def screening_summary(crosswalk, total, passed, merged, final, full, qc_pass):
    """Report all 23 communities, including absent, unmapped, and screened-out cases."""
    rows = []
    for info in crosswalk.to_dict('records'):
        comm = info['community_type']
        idx = merged.index[merged.community_type == comm]
        sub = final[final.community_type == comm]
        row = dict(info, input_grid_community_rows=total[comm],
                   after_lulcc_screen=passed[comm],
                   complete_gfed=int(full.loc[idx].sum()),
                   qc_flagged_complete_gfed=int((full.loc[idx] & ~qc_pass.loc[idx]).sum()),
                   after_fire_screen=len(sub), n_selected=0)
        if not info['selection_enabled']:
            row['selection_status'] = info['mapping_status']
        elif total[comm] == 0:
            row['selection_status'] = 'no_lulcc_records'
        elif passed[comm] == 0:
            row['selection_status'] = 'failed_lulcc_screen'
        elif not row['complete_gfed']:
            row['selection_status'] = 'failed_gfed_completeness'
        elif sub.empty:
            row['selection_status'] = 'failed_fire_or_optional_qc_screen'
        else:
            row['selection_status'] = 'ready_for_selection'
        rows.append(row)
    return pd.DataFrame(rows)

try:
    summary = screening_summary(crosswalk, total, passed, merged, final, full, qc_pass)
    winners, rankings = [], []
    # Check every community; disabled communities remain documented in the summary.
    for comm in range(1, 24):
        sub = final[final.community_type == comm].copy()
        if sub.empty:
            continue
        used = set()
        for label, q in TARGET_QUANTILES.items():
            best, short = select_group(sub, comm, label, q, used)
            if not short.empty:
                rankings.append(short)
            if best is None:
                continue
            best['selection_method'] = 'automatic'
            best['selection_reason'] = 'GFED percentile band + community dominance/stability'
            best['manual_override_outside_quantile_bin'] = False
            winners.append(best)
            used.add((float(best.iloc[0].lon), float(best.iloc[0].lat)))
    # Write headers even when empty; do not invent selections for missing fire groups.
    extra = ['target_fire_fraction', 'distance_to_target', 'distance_to_target_normalized',
             'fire_group', 'target_quantile', 'shortlist_rank', 'fallback_no_quantile_bin']
    if rankings:
        shortlist = pd.concat(rankings, ignore_index=True)
    else:
        shortlist = pd.DataFrame(columns=list(final.columns) + extra)

    if winners:
        selected = pd.concat(winners, ignore_index=True)
    else:
        selected = pd.DataFrame(columns=list(shortlist.columns) + [
            'selection_method', 'selection_reason', 'manual_override_outside_quantile_bin',
        ])
    shortlist.to_csv(paths['shortlist'], index=False)
except Exception as exc:
    manifest.update(status='failed', error=str(exc))
    save_run_summary(manifest, paths)
    raise
else:
    save_run_summary(manifest, paths)

display(shortlist.head())
display(selected)


## 6. Apply optional manual overrides and save final results

On the first run, this step creates an empty `03_manual_overrides.csv` in the notebook result directory. To replace an automatic point, enter `community_type`, `fire_group`, `lon`, `lat`, and a nonempty scientific `reason`. Then restart the kernel and run all cells again.

The replacement must be in the eligible candidate table and correspond to an existing selected group. Duplicate coordinates within a community are rejected. The replacement inherits the group's target, receives updated distance and bin diagnostics, and has no automatic shortlist rank. The ranked-shortlist file continues to describe automatic selection.

The summary reports every community, even when no point can be selected. Ordinary `if/elif/else` blocks replace nested conditional expressions in manual-bin checks and final run-status assignment.


In [ ]:
def apply_overrides(selected, candidates, path):
    """Apply optional manual replacements and recalculate their selection diagnostics."""
    template = ['community_type', 'fire_group', 'lon', 'lat', 'reason']
    if not path.exists():
        pd.DataFrame(columns=template).to_csv(path, index=False)
        return selected
    override = pd.read_csv(path)
    if not set(template).issubset(override.columns):
        raise ValueError(f'The manual override table requires columns: {template}')
    if override.duplicated(['community_type', 'fire_group']).any():
        raise ValueError('Only one override is allowed per community/fire_group.')
    out = selected.copy()
    out['manual_override_outside_quantile_bin'] = False
    for row in override.itertuples(index=False):
        if pd.isna(row.community_type) or float(row.community_type) % 1:
            raise ValueError('Manual override community_type must be an integer.')
        comm, label = int(row.community_type), str(row.fire_group)
        if label not in TARGET_QUANTILES or not isinstance(row.reason, str) or not row.reason.strip():
            raise ValueError('An override requires a valid fire_group and a nonempty scientific reason.')
        old = out[(out.community_type == comm) & (out.fire_group == label)]
        replacement = candidates[(candidates.community_type == comm) &
                                  (candidates.lon == round(float(row.lon), 6)) &
                                  (candidates.lat == round(float(row.lat), 6))]
        if len(old) != 1 or len(replacement) != 1:
            raise ValueError(f'The replacement is ineligible or the original selection does not exist: {comm}, {label}, {row.lon}, {row.lat}')
        rep = replacement.iloc[0].copy()
        for col in ('fire_group', 'target_quantile', 'target_fire_fraction'):
            rep[col] = old.iloc[0][col]
        rep['distance_to_target'] = abs(rep.gfed_mean_grid_fraction - rep.target_fire_fraction)
        rep['distance_to_target_normalized'] = rep.distance_to_target / max(rep.target_fire_fraction, 1e-12)
        # A manual replacement may be outside the shortlist, so it has no automatic rank.
        rep['shortlist_rank'] = np.nan
        rep['fallback_no_quantile_bin'] = old.iloc[0]['fallback_no_quantile_bin']
        p = rep.empirical_percentile
        if label == 'low_fire':
            in_bin = p < 0.40
        elif label == 'medium_fire':
            in_bin = 0.40 <= p < 0.60
        else:
            in_bin = p >= 0.60
        rep['manual_override_outside_quantile_bin'] = not in_bin
        rep['selection_method'] = 'manual_override'
        rep['selection_reason'] = row.reason.strip()
        for col in out.columns:
            if col in rep.index:
                out.loc[old.index[0], col] = rep[col]
    if out.duplicated(KEY).any():
        raise ValueError('Manual overrides reuse a grid cell within the same community.')
    return out

try:
    selected = apply_overrides(selected, final, paths['overrides'])
    selected = selected.sort_values(['community_type', 'target_quantile'])
    selected.to_csv(paths['selected'], index=False)
    for comm, n in selected.community_type.value_counts().items():
        where = summary.community_type == comm
        summary.loc[where, 'n_selected'] = int(n)
        if n == 3:
            summary.loc[where, 'selection_status'] = 'selected_3_groups'
        else:
            summary.loc[where, 'selection_status'] = 'selected_fewer_than_3'
    summary.to_csv(paths['summary'], index=False)
    n_review = int(selected.gfed_area_needs_review.sum()) if 'gfed_area_needs_review' in selected else 0
    if selected.empty:
        summary_note = 'no_selection'
    elif n_review > 0:
        summary_note = 'complete_with_qc_flags'
    else:
        summary_note = 'complete'
    manifest.update(status=summary_note, n_candidates=len(final), n_selected=len(selected),
                    n_selected_needing_area_review=n_review,
                    n_matched_candidates_needing_area_review=len(qc_rows))
    print(summary[['community_type', 'community_name', 'after_lulcc_screen',
                   'after_fire_screen', 'n_selected', 'selection_status']].to_string(index=False))
    if n_review:
        print(f'Review required: {n_review} selected points have large annual GFED area flags.')
    print('Stage 3 output directory: ', RESULT_DIR, flush=True)
except Exception as exc:
    manifest.update(status='failed', error=str(exc))
    save_run_summary(manifest, paths)
    raise
else:
    save_run_summary(manifest, paths)

display(selected)
display(summary)


## Pipeline outputs

All paths are listed in `03_run_summary.json` under `RESULT_DIR`.

| Output | Contents |
| --- | --- |
| `03_community_gfed_crosswalk.csv` | All 23 communities, class mappings, statuses, and explanatory notes |
| `03_missing_gfed_summary_rows.csv` | Required GFED summary records that are absent |
| `03_gfed_area_review.csv` | Matched candidates flagged for annual-area review |
| `03_all_eligible_candidates.csv` | Candidates passing the configured screens, with empirical percentiles |
| `03_ranked_shortlists.csv` | Automatic ranked shortlists for each selected fire group |
| `03_selected_observation_only.csv` | Final selections after optional manual overrides |
| `03_community_screening_summary.csv` | Screening counts and selection status for all 23 communities |
| `03_manual_overrides.csv` | Preserved user-supplied replacements, or an empty template |
| `03_run_summary.json` | Inputs, settings, output locations, and current run status |

The final selection contains up to one low-, one medium-, and one high-fire cell per enabled community. Communities without eligible candidates remain in the summary.

### Changes from the source script

- Translate all comments, docstrings, progress messages, and error messages into English.
- Replace the single `main()` workflow with numbered, sequential notebook steps and table previews.
- Fix `mapping_info()` to return both the expected status and note.
- Replace row-wise `apply(lambda ...)`, the GFED-class-count lambda, and nested conditional expressions with explicit loops, a lookup dictionary, and `if/elif/else` blocks.
- Keep CSV chunking, temporary SQLite aggregation, validation rules, proxy assignments, selection thresholds, quantile targets, bin boundaries, and shortlist ranking rules.
- Add intermediate run-summary checkpoints. Use a separate `_notebook` output directory.
- Retain the legacy `gfed_ba_total_21y_km2` field for compatibility; its values follow the configured year range even if that range is not 21 years.

Run from the beginning after changing any setting or manual override. The original server inputs are needed for a full end-to-end run.

### Verification

All code cells passed Python syntax checks. Six synthetic scenarios were checked: ordinary selection (including small communities and fallback), a manual override outside its bin, no eligible candidates, optional QC exclusion, missing GFED summary rows, and inconsistent annual totals. Successful CSV outputs matched the source script after repairing only its mapping return-value defect in memory. Both invalid-input scenarios stopped with a failed run summary.

The original source file was verified unchanged. No server input files were read and no real calibration cells were selected during these checks.
